# Hands-on: PSO Portfolio Optimization with Real IDX Stock Data

**BBCA.JK and TLKM.JK | 2025 | Educational demonstration**

Run cells in order. Internet is required to download market data. Historical performance is not a forecast.

## Step 1 — Install and import packages
We implement PSO manually; no PSO library is required.

In [ ]:
%pip -q install yfinance pandas numpy matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

## Step 2 — Download actual adjusted daily prices
Use **auto-adjusted Close** (split/dividend-adjusted) from Yahoo Finance. Fixed dates make classroom results more reproducible; data vendors may revise histories. If download fails, retry or use the optional CSV fallback.

In [ ]:
tickers = ["BBCA.JK", "TLKM.JK"]
raw = yf.download(tickers, start="2025-01-01", end="2026-01-01", auto_adjust=True, progress=False, threads=False)
if raw.empty:
    raise RuntimeError("No market data returned. Check internet access or Yahoo Finance availability.")
prices = raw["Close"].reindex(columns=tickers).dropna()
if len(prices) < 100:
    raise RuntimeError(f"Only {len(prices)} aligned price rows. Inspect download before continuing.")
print("Dates:", prices.index.min().date(), "to", prices.index.max().date())
print("Observations:", len(prices))
display(prices.head())

## Step 3 — Compute daily returns and annualized statistics

Daily simple return: $r_{i,t}=P_{i,t}/P_{i,t-1}-1$. Assume 252 trading days per year. $\mu=252\bar r$ and $\Sigma=252\operatorname{Cov}(r)$.

In [ ]:
returns = prices.pct_change().dropna()
mu = returns.mean().to_numpy() * 252
Sigma = returns.cov().to_numpy() * 252
print("Annualized mean returns:", dict(zip(tickers, np.round(mu, 4))))
print("Annualized covariance matrix:\n", Sigma)
print("Correlation:\n", returns.corr())

## Step 4 — Decision variable and objective

Let $x$ be the BBCA weight, and $1-x$ the TLKM weight, with $0\le x\le1$. Maximize $f(x)=\mathbf w^T\mu-\lambda\mathbf w^T\Sigma\mathbf w$, using $\lambda=3$. Both return and variance are annualized.

In [ ]:
risk_penalty = 3.0
def portfolio_metrics(x):
    x = float(np.asarray(x).item())
    w = np.array([x, 1.0-x])
    expected_return = float(w @ mu)
    variance = float(w @ Sigma @ w)
    return expected_return, variance, np.sqrt(max(variance, 0.0))

def fitness(x):
    ret, var, _ = portfolio_metrics(x)
    return ret - risk_penalty * var

for x in [0, .25, .5, .75, 1]:
    print(f"BBCA={x:.0%}, TLKM={1-x:.0%}, fitness={fitness(x):.5f}")

## Step 5 — Visualize the objective
A one-dimensional grid makes it easy to understand what PSO is searching for.

In [ ]:
grid = np.linspace(0, 1, 501)
values = np.array([fitness(x) for x in grid])
plt.plot(grid, values)
plt.xlabel("Weight of BBCA")
plt.ylabel("Risk-return fitness")
plt.title("Objective function on real 2025 stock data")
plt.grid(alpha=.3)
plt.show()

## Step 6 — PSO mechanics
Each particle is a candidate BBCA weight. Update $v_i\leftarrow\omega v_i+c_1r_1(pbest_i-x_i)+c_2r_2(gbest-x_i)$, then $x_i\leftarrow\mathrm{clip}(x_i+v_i,0,1)$. The best-so-far global fitness is tracked at every iteration.

In [ ]:
rng = np.random.default_rng(42)
n_particles, iterations = 8, 60
omega, c1, c2 = 0.7, 1.5, 1.5
x = rng.uniform(0, 1, n_particles)
v = np.zeros(n_particles)
pbest = x.copy()
pbest_scores = np.array([fitness(z) for z in x])
best_idx = int(np.argmax(pbest_scores))
gbest = float(pbest[best_idx])
gbest_score = float(pbest_scores[best_idx])
history = [gbest_score]
print("Initial particle positions:", np.round(x, 3))
print("Initial global best:", gbest, gbest_score)

## Step 7 — Run PSO (from scratch)

In [ ]:
for t in range(iterations):
    r1 = rng.random(n_particles)
    r2 = rng.random(n_particles)
    v = omega*v + c1*r1*(pbest-x) + c2*r2*(gbest-x)
    x = np.clip(x+v, 0, 1)
    scores = np.array([fitness(z) for z in x])
    improved = scores > pbest_scores
    pbest[improved] = x[improved]
    pbest_scores[improved] = scores[improved]
    idx = int(np.argmax(pbest_scores))
    if pbest_scores[idx] > gbest_score:
        gbest = float(pbest[idx])
        gbest_score = float(pbest_scores[idx])
    history.append(gbest_score)

ret, var, vol = portfolio_metrics(gbest)
print(f"Optimal BBCA weight: {gbest:.2%}")
print(f"Optimal TLKM weight: {1-gbest:.2%}")
print(f"Expected annual return: {ret:.2%}")
print(f"Annualized volatility: {vol:.2%}")
print(f"Objective value: {gbest_score:.6f}")

## Step 8 — Visualize convergence and compare with a grid-search check
Grid search is for **validation**, not the PSO optimization algorithm.

In [ ]:
fig, ax = plt.subplots()
ax.plot(history)
ax.set(xlabel="Iteration", ylabel="Best-so-far fitness", title="PSO convergence")
ax.grid(alpha=.3)
plt.show()
grid_best_x = float(grid[np.argmax(values)])
print(f"PSO weight BBCA: {gbest:.4f}; grid-search check: {grid_best_x:.4f}")
print(f"Absolute fitness gap: {max(values)-gbest_score:.8f}")

## Step 9 — Classroom experiments
1. Change `risk_penalty` to 1, 5, 10 and rerun Steps 5–8. How do optimal weights change?
2. Change `n_particles` to 3 and 30. Does convergence change?
3. Change `omega` to 0.3 or 0.9. What happens?
4. Why is the global-best fitness curve nondecreasing?
5. Compare optimized portfolio to a 50/50 allocation.

**Note:** The objective can favor a boundary allocation (100% in one stock). This is mathematically valid under the stated constraints. Historical returns are estimates, not guarantees.